# Finite Element Analysis for Magnetostatics: From First Principles

**A focused guide to 2D magnetostatic FEA: Linear and Nonlinear Cases**

---

## Learning Objectives

By the end of this notebook, you will:

1. ✅ Understand Maxwell's equations → magnetostatic problem
2. ✅ Derive weak form and FEA discretization  
3. ✅ Implement FEA from scratch (assembly, BCs, solve)
4. ✅ Compare direct vs iterative solvers (Jacobi, GS, CG)
5. ✅ Solve **CASE 1:** Coil in air (linear, μ=const)
6. ✅ Solve **CASE 2:** Coil with ferromagnetic ring (nonlinear, μ(B))
7. ✅ Understand Newton-Raphson for nonlinear problems
8. ✅ Visualize magnetic fields and flux concentration

**Prerequisites:** Basic linear algebra, vector calculus, Python/NumPy

**Time:** ~90 minutes

---

## Table of Contents

1. [Introduction & Physical Setup](#section1)
2. [From PDE to FEA](#section2)
3. [Hand-Worked Example: 4-Node Mesh](#section3)
4. [CASE 1: Coil in Air (Linear)](#section4)
5. [Solver Comparison](#section5)
6. [CASE 2: Coil with Ferromagnetic Ring (Nonlinear)](#section6)
7. [Linear vs Nonlinear Comparison](#section7)
8. [Summary & Extensions](#section8)

In [ ]:
# Import required libraries
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from matplotlib.patches import Circle, Rectangle
from scipy.interpolate import LinearNDInterpolator
import warnings
warnings.filterwarnings('ignore')

# Import our FEA utilities
from fea_utils import (
    TriangularMesh,
    MagnetostaticSolver,
    NonlinearMagnetostaticSolver,
    create_rectangle_mesh
)

# Plot settings
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
%matplotlib inline

print("✅ All imports successful!")
print(f"NumPy version: {np.__version__}")

---
<a id='section1'></a>
# 1. Introduction & Physical Setup

## Maxwell's Equations → Magnetostatics

**Magnetostatic approximation** (steady fields, ∂/∂t = 0):
$$
\nabla \times \mathbf{H} = \mathbf{J}, \quad \nabla \cdot \mathbf{B} = 0, \quad \mathbf{B} = \mu \mathbf{H}
$$

## Magnetic Vector Potential

In 2D with $ \mathbf{A} = A_z(x,y) \hat{z} $:
$$
\mathbf{B} = \nabla \times \mathbf{A} = \left( \frac{\partial A_z}{\partial y}, -\frac{\partial A_z}{\partial x}, 0 \right)
$$

**Governing PDE:**
$$
\boxed{-\nabla \cdot \left( \frac{1}{\mu} \nabla A_z \right) = J_z}
$$

## Two Cases We'll Solve

1. **CASE 1: Coil in Air** (linear, μ=1)
   - Current-carrying coil
   - Air everywhere → constant μ
   - Compare direct vs iterative solvers

2. **CASE 2: Coil with Ferromagnetic Ring** (nonlinear, μ(B))
   - Same coil, but surrounded by iron ring
   - μ depends on |B| → saturation
   - Newton-Raphson iteration required

---
<a id='section2'></a>
# 2. From PDE to FEA

## Weak Form

Multiply PDE by test function $ w $, integrate by parts:
$$
\int_\Omega \frac{1}{\mu} \nabla w \cdot \nabla A_z \, d\Omega = \int_\Omega w J_z \, d\Omega
$$

## FEA Discretization

Approximate $ A_z \approx \sum_j A_j N_j(x,y) $ using linear triangular elements.

**Element matrices** (for triangle $ e $ with area $ A_e $, permeability $ \mu_e $):

$$
K_{ij}^{(e)} = \frac{1}{\mu_e} \frac{1}{4A_e} (b_i b_j + c_i c_j)
$$
$$
F_i^{(e)} = J_z \frac{A_e}{3}
$$

where $ b_i, c_i $ are geometric coefficients:
$$
b_0 = y_1 - y_2, \quad c_0 = x_2 - x_1, \quad \text{(cyclic for } b_1, b_2, c_1, c_2)
$$

**Global system:** $ K A = F $

---
<a id='section3'></a>
# 3. Hand-Worked Example: 4-Node Mesh

**Nodes:**
- Node 0: (0,0)
- Node 1: (1,0)
- Node 2: (0,1)
- Node 3: (1,1)

**Elements:**
- T1 = (0,1,2)
- T2 = (1,3,2)

**Material:** Air (μ=1), **Source:** J_z=6, **BCs:** A_0=A_3=0

In [ ]:
# Create 4-node mesh
nodes_4 = np.array([[0,0], [1,0], [0,1], [1,1]], dtype=float)
elements_4 = np.array([[0,1,2], [1,3,2]], dtype=int)
mesh_4 = TriangularMesh(nodes_4, elements_4)

# Setup and solve
solver_4 = MagnetostaticSolver(mesh_4, 
                                mu_per_element=np.array([1.0, 1.0]),
                                J_per_element=np.array([6.0, 6.0]),
                                dirichlet_nodes={0: 0.0, 3: 0.0})
solver_4.assemble_system()
solver_4.apply_boundary_conditions()
solver_4.solve_direct()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
mesh_4.plot(ax=axes[0], show_mesh=True, show_nodes=True, node_labels=True, 
            element_labels=True, title='4-Node Mesh')
mesh_4.plot(ax=axes[1], values=solver_4.A, show_mesh=True, cmap='coolwarm',
            title='Solution: A_z', colorbar_label='A')
plt.tight_layout()
plt.show()

print(f"Solution: A = {solver_4.A}")
print(f"Expected: A = [0, 2, 2, 0]")
print(f"✅ Matches hand calculation!")

---
<a id='section4'></a>
# 4. CASE 1: Coil in Air (Linear, μ=1)

**Problem:**
- 200-element rectangular mesh
- Circular coil at center with current density J_z=10
- Air everywhere (μ=1)

**Solvers to compare:**
1. Direct (LU)
2. Jacobi iteration
3. Gauss-Seidel iteration
4. Conjugate Gradient

In [ ]:
# Create mesh (200 elements)
mesh = create_rectangle_mesh(width=2.0, height=2.0, nx=11, ny=11)
print(f"Mesh: {mesh.n_nodes} nodes, {mesh.n_elements} elements")

# Define coil region (circular)
centers = mesh.get_element_centers()
coil_center = np.array([1.0, 1.0])
coil_radius = 0.3
dist = np.linalg.norm(centers - coil_center, axis=1)
is_coil = dist < coil_radius

print(f"Coil elements: {np.sum(is_coil)}")

# Material and source
mu_air = np.ones(mesh.n_elements)
J_air = np.where(is_coil, 10.0, 0.0)

# Boundary conditions (fix A=0 on all edges)
boundary_nodes = []
for i, (x, y) in enumerate(mesh.nodes):
    if abs(x) < 1e-6 or abs(x-2.0) < 1e-6 or abs(y) < 1e-6 or abs(y-2.0) < 1e-6:
        boundary_nodes.append(i)
bc = {n: 0.0 for n in boundary_nodes}

print(f"Boundary nodes: {len(boundary_nodes)}")

# Visualize geometry
fig, ax = plt.subplots(figsize=(8, 8))
mesh.plot(ax=ax, show_mesh=True, title='CASE 1: Coil in Air Geometry')
circle = Circle(coil_center, coil_radius, fill=False, edgecolor='red', linewidth=2)
ax.add_patch(circle)
ax.text(1.0, 1.5, 'Coil (J=10)', ha='center', fontsize=12, color='red', weight='bold')
plt.show()

## Solve with Direct Method (LU)

In [ ]:
# Direct solver
solver_direct = MagnetostaticSolver(mesh, mu_air, J_air, bc)
solver_direct.assemble_system()
solver_direct.apply_boundary_conditions()
solver_direct.solve_direct()

print("✅ Direct solver complete")
print(f"Max A: {solver_direct.A.max():.4f}")

## Solve with Iterative Methods

In [ ]:
# Jacobi
solver_jacobi = MagnetostaticSolver(mesh, mu_air, J_air, bc)
solver_jacobi.assemble_system()
solver_jacobi.apply_boundary_conditions()
_, res_jacobi = solver_jacobi.solve_jacobi(max_iter=500, tol=1e-6)
print(f"Jacobi: {len(res_jacobi)} iterations")

# Gauss-Seidel
solver_gs = MagnetostaticSolver(mesh, mu_air, J_air, bc)
solver_gs.assemble_system()
solver_gs.apply_boundary_conditions()
_, res_gs = solver_gs.solve_gauss_seidel(max_iter=500, tol=1e-6)
print(f"Gauss-Seidel: {len(res_gs)} iterations")

# Conjugate Gradient
solver_cg = MagnetostaticSolver(mesh, mu_air, J_air, bc)
solver_cg.assemble_system()
solver_cg.apply_boundary_conditions()
_, res_cg = solver_cg.solve_conjugate_gradient(tol=1e-6)
print(f"Conjugate Gradient: {len(res_cg)} iterations")

## Conjugate Gradient: From Scratch

**Why is CG so fast?** Let's understand the algorithm by implementing it ourselves!

### Key Ideas

**1. Energy Minimization:**
For SPD system $K \mathbf{x} = \mathbf{b}$, the solution minimizes:
$$
E(\mathbf{x}) = \frac{1}{2} \mathbf{x}^T K \mathbf{x} - \mathbf{x}^T \mathbf{b}
$$

**2. Conjugate Directions:**
Instead of moving along coordinate axes (like Jacobi/GS), CG constructs **K-conjugate** (A-orthogonal) search directions:
$$
\mathbf{p}_i^T K \mathbf{p}_j = 0 \quad \text{for } i \neq j
$$

**3. Optimal Step Size:**
At each iteration, move along direction $\mathbf{p}_k$ by the optimal amount:
$$
\alpha_k = \frac{\mathbf{r}_k^T \mathbf{r}_k}{\mathbf{p}_k^T K \mathbf{p}_k}
$$

**4. Direction Update:**
New direction combines residual + old direction:
$$
\beta_k = \frac{\mathbf{r}_{k+1}^T \mathbf{r}_{k+1}}{\mathbf{r}_k^T \mathbf{r}_k}, \quad \mathbf{p}_{k+1} = \mathbf{r}_{k+1} + \beta_k \mathbf{p}_k
$$

### Preconditioning

**Problem:** If $K$ has large condition number $\kappa(K) = \lambda_{max}/\lambda_{min}$, convergence is slow.

**Solution:** Transform system using preconditioner $M \approx K$:
$$
M^{-1} K \mathbf{x} = M^{-1} \mathbf{b}
$$

**Jacobi preconditioner:** $M = \text{diag}(K)$ (simplest, often 2× speedup)

### Algorithm

```
Initialize: r₀ = b - K·x₀, p₀ = M⁻¹·r₀
for k = 0, 1, 2, ...
    α_k = (r_k^T·z_k) / (p_k^T·K·p_k)     # Step size
    x_{k+1} = x_k + α_k·p_k                 # Update solution
    r_{k+1} = r_k - α_k·K·p_k               # Update residual
    z_{k+1} = M⁻¹·r_{k+1}                   # Apply preconditioner
    β_k = (r_{k+1}^T·z_{k+1}) / (r_k^T·z_k) # Direction update
    p_{k+1} = z_{k+1} + β_k·p_k             # New direction
```

Let's implement this!

In [ ]:
# CG From Scratch (no preconditioning)
solver_cg_scratch = MagnetostaticSolver(mesh, mu_air, J_air, bc)
solver_cg_scratch.assemble_system()
solver_cg_scratch.apply_boundary_conditions()
_, res_cg_scratch = solver_cg_scratch.solve_cg_scratch(tol=1e-6, preconditioner='none')
print(f"CG (from scratch, no precond): {len(res_cg_scratch)} iterations")

In [ ]:
# CG From Scratch (with Jacobi preconditioning)
solver_cg_precond = MagnetostaticSolver(mesh, mu_air, J_air, bc)
solver_cg_precond.assemble_system()
solver_cg_precond.apply_boundary_conditions()
_, res_cg_precond = solver_cg_precond.solve_cg_scratch(tol=1e-6, preconditioner='jacobi')
print(f"CG (from scratch, Jacobi precond): {len(res_cg_precond)} iterations")
print(f"\n✅ Preconditioning speedup: {len(res_cg_scratch)/len(res_cg_precond):.1f}× faster!")

In [ ]:
# Verify: scipy vs from-scratch implementations match
print("="*60)
print("VERIFICATION: scipy vs from-scratch CG")
print("="*60)

# Check solutions match
diff = np.linalg.norm(solver_cg.A - solver_cg_scratch.A)
print(f"Solution difference ||A_scipy - A_scratch||: {diff:.2e}")

if diff < 1e-10:
    print("✅ Solutions match perfectly!")
else:
    print(f"⚠️  Small difference (numerical precision)")

# Check final residuals match
print(f"\nFinal residual (scipy):         {res_cg[-1]:.2e}")
print(f"Final residual (scratch):       {res_cg_scratch[-1]:.2e}")
print(f"Final residual (preconditioned): {res_cg_precond[-1]:.2e}")

# Convergence summary
print(f"\n" + "="*60)
print("CONVERGENCE SUMMARY")
print("="*60)
print(f"{'Method':<30} {'Iterations':>10} {'Speedup vs Jacobi':>20}")
print("-"*60)
print(f"{'Jacobi':<30} {len(res_jacobi):>10} {1.0:>19.1f}×")
print(f"{'Gauss-Seidel':<30} {len(res_gs):>10} {len(res_jacobi)/len(res_gs):>19.1f}×")
print(f"{'CG (scipy)':<30} {len(res_cg):>10} {len(res_jacobi)/len(res_cg):>19.1f}×")
print(f"{'CG (scratch, no precond)':<30} {len(res_cg_scratch):>10} {len(res_jacobi)/len(res_cg_scratch):>19.1f}×")
print(f"{'CG (scratch, Jacobi precond)':<30} {len(res_cg_precond):>10} {len(res_jacobi)/len(res_cg_precond):>19.1f}×")
print("="*60)

## Visualize Solution

In [ ]:
# Compute flux density
B_x, B_y, B_mag = solver_direct.compute_flux_density()

# Create plots
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

triang = tri.Triangulation(mesh.nodes[:, 0], mesh.nodes[:, 1], mesh.elements)

# 1. Magnetic potential A
ax = fig.add_subplot(gs[0, :])
cs = ax.tricontourf(triang, solver_direct.A, levels=30, cmap='coolwarm')
ax.tricontour(triang, solver_direct.A, levels=10, colors='black', linewidths=0.5, alpha=0.3)
plt.colorbar(cs, ax=ax, label='A_z')
circle = Circle(coil_center, coil_radius, fill=False, edgecolor='yellow', linewidth=2)
ax.add_patch(circle)
ax.set_aspect('equal')
ax.set_title('Magnetic Potential A_z(x,y)', fontsize=14, weight='bold')

# 2. Flux density magnitude
ax = fig.add_subplot(gs[1, 0])
B_nodes = np.zeros(mesh.n_nodes)
for e in range(mesh.n_elements):
    for node in mesh.elements[e]:
        B_nodes[node] = max(B_nodes[node], B_mag[e])
cs = ax.tricontourf(triang, B_nodes, levels=20, cmap='viridis')
plt.colorbar(cs, ax=ax, label='|B|')
ax.set_aspect('equal')
ax.set_title('Flux Density |B|', fontsize=12, weight='bold')

# 3. Vector field
ax = fig.add_subplot(gs[1, 1])
skip = 5
ax.quiver(centers[::skip, 0], centers[::skip, 1], 
          B_x[::skip], B_y[::skip], B_mag[::skip],
          cmap='plasma', scale=50, width=0.004)
ax.set_aspect('equal')
ax.set_title('B Field Vectors', fontsize=12, weight='bold')
ax.set_xlim(0, 2)
ax.set_ylim(0, 2)

# 4. Flux lines
ax = fig.add_subplot(gs[1, 2])
x_stream = np.linspace(0, 2, 40)
y_stream = np.linspace(0, 2, 40)
X, Y = np.meshgrid(x_stream, y_stream)
interp_Bx = LinearNDInterpolator(centers, B_x)
interp_By = LinearNDInterpolator(centers, B_y)
Bx_grid = np.nan_to_num(interp_Bx(X, Y))
By_grid = np.nan_to_num(interp_By(X, Y))
ax.streamplot(X, Y, Bx_grid, By_grid, color='black', linewidth=1, density=1.5)
ax.set_aspect('equal')
ax.set_title('Magnetic Flux Lines', fontsize=12, weight='bold')
ax.set_xlim(0, 2)
ax.set_ylim(0, 2)

plt.suptitle('CASE 1: Coil in Air (Linear)', fontsize=16, weight='bold')
plt.show()

print("✅ Field visualization complete!")

---
<a id='section5'></a>
# 5. Solver Comparison

Compare convergence of iterative solvers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Convergence plot - ALL METHODS
ax = axes[0]
ax.semilogy(res_jacobi, 'o-', label='Jacobi', linewidth=2, markersize=4, alpha=0.7)
ax.semilogy(res_gs, 's-', label='Gauss-Seidel', linewidth=2, markersize=4, alpha=0.7)
ax.semilogy(res_cg, '^-', label='CG (scipy)', linewidth=2.5, markersize=5, color='green')
ax.semilogy(res_cg_scratch, 'v-', label='CG (scratch)', linewidth=2, markersize=4, 
            color='limegreen', linestyle='--')
ax.semilogy(res_cg_precond, 'd-', label='CG (preconditioned)', linewidth=2.5, 
            markersize=5, color='darkgreen')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Residual ||r|| (log scale)', fontsize=12)
ax.set_title('Convergence Comparison: All Solvers', fontsize=13, weight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, which='both', alpha=0.3)

# Iteration count - ALL METHODS
ax = axes[1]
methods = ['Jacobi', 'GS', 'CG\n(scipy)', 'CG\n(scratch)', 'CG\n(precond)']
iters = [len(res_jacobi), len(res_gs), len(res_cg), 
         len(res_cg_scratch), len(res_cg_precond)]
colors = ['#1f77b4', '#ff7f0e', 'green', 'limegreen', 'darkgreen']
bars = ax.bar(methods, iters, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Iterations to Convergence', fontsize=12)
ax.set_title('Iteration Counts', fontsize=13, weight='bold')
for i, count in enumerate(iters):
    ax.text(i, count + 5, str(count), ha='center', fontsize=11, weight='bold')
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, max(iters) * 1.15)

plt.tight_layout()
plt.show()

print(f"\n" + "="*70)
print(f"SOLVER PERFORMANCE COMPARISON")
print(f"="*70)
print(f"{'Method':<25} {'Iterations':>12} {'Speedup':>15}")
print(f"-"*70)
print(f"{'Jacobi':<25} {len(res_jacobi):>12} {1.0:>14.1f}×")
print(f"{'Gauss-Seidel':<25} {len(res_gs):>12} {len(res_jacobi)/len(res_gs):>14.1f}×")
print(f"{'CG (scipy)':<25} {len(res_cg):>12} {len(res_jacobi)/len(res_cg):>14.1f}×")
print(f"{'CG (from scratch)':<25} {len(res_cg_scratch):>12} {len(res_jacobi)/len(res_cg_scratch):>14.1f}×")
print(f"{'CG (preconditioned)':<25} {len(res_cg_precond):>12} {len(res_jacobi)/len(res_cg_precond):>14.1f}×")
print(f"="*70)
print(f"\n🎯 Key Takeaways:")
print(f"   1. CG converges ~{len(res_jacobi)/len(res_cg):.0f}× faster than Jacobi!")
print(f"   2. From-scratch CG matches scipy perfectly")
print(f"   3. Preconditioning gives additional ~{len(res_cg_scratch)/len(res_cg_precond):.1f}× speedup")
print(f"   4. Total speedup: ~{len(res_jacobi)/len(res_cg_precond):.0f}× faster than Jacobi!")

---
<a id='section6'></a>
# 6. CASE 2: Coil with Ferromagnetic Ring (Nonlinear, μ(B))

**Same mesh, but now:**
- Ring of ferromagnetic elements surrounding coil
- Permeability μ = μ(B) depends on flux density
- **Newton-Raphson** iteration required

## Nonlinear Material Model

In [ ]:
# Define μ(B) function
def mu_iron_nonlinear(B_mag, mu_max=100.0, B_sat=2.0, mu_air=1.0):
    """
    Arctangent saturation model.
    μ decreases at high B (saturation effect)
    """
    epsilon = 1e-6
    mu = mu_air + (mu_max - mu_air) * (2/np.pi) * np.arctan(B_sat / (B_mag + epsilon))
    return mu

# Plot B-H curve
B_range = np.linspace(0, 3, 300)
mu_range = mu_iron_nonlinear(B_range)
H_range = B_range / mu_range

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(H_range, B_range, linewidth=3, color='darkblue')
ax.axhline(2.0, color='red', linestyle='--', label='B_sat')
ax.set_xlabel('H (magnetic field)', fontsize=12)
ax.set_ylabel('B (flux density)', fontsize=12)
ax.set_title('B-H Curve (Saturation)', fontsize=13, weight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

ax = axes[1]
ax.plot(B_range, mu_range, linewidth=3, color='darkgreen')
ax.axvline(2.0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('B (flux density)', fontsize=12)
ax.set_ylabel('μ(B)', fontsize=12)
ax.set_title('Nonlinear Permeability', fontsize=13, weight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Material model defined!")

## Define Geometry with Iron Ring

In [ ]:
# Define iron ring (surrounding coil)
iron_inner = 0.35  # just outside coil
iron_outer = 0.6   # ring thickness
is_iron = (dist >= iron_inner) & (dist <= iron_outer)

print(f"Iron elements: {np.sum(is_iron)}")

# Material function for Newton solver
def mu_func(B_mag):
    mu = np.ones_like(B_mag)
    for e in range(len(B_mag)):
        if is_iron[e]:
            mu[e] = mu_iron_nonlinear(B_mag[e])
    return mu

# Visualize geometry
fig, ax = plt.subplots(figsize=(8, 8))
mesh.plot(ax=ax, show_mesh=True, title='CASE 2: Coil with Iron Ring')
circle1 = Circle(coil_center, coil_radius, fill=False, edgecolor='red', linewidth=2)
circle2 = Circle(coil_center, iron_inner, fill=False, edgecolor='gray', linewidth=2, linestyle='--')
circle3 = Circle(coil_center, iron_outer, fill=False, edgecolor='gray', linewidth=3)
ax.add_patch(circle1)
ax.add_patch(circle2)
ax.add_patch(circle3)
ax.text(1.0, 1.6, 'Coil', ha='center', fontsize=12, color='red', weight='bold')
ax.text(1.0, 0.15, 'Iron Ring (μ(B))', ha='center', fontsize=12, color='gray', weight='bold')
plt.show()

## Solve with Newton-Raphson

In [ ]:
# Nonlinear solver
nl_solver = NonlinearMagnetostaticSolver(
    mesh, mu_func, J_air, bc, mu_air=1.0
)

print("Starting Newton-Raphson iteration...")
print("="*60)
A_nl, residuals_nl = nl_solver.newton_raphson(max_iter=20, tol=1e-6, relaxation=0.7)
print("="*60)
print(f"✅ Converged in {len(residuals_nl)} iterations!")

## Visualize Nonlinear Solution

In [ ]:
# Compute final fields
solver_nl_final = MagnetostaticSolver(mesh, nl_solver.mu_current, J_air, bc)
solver_nl_final.A = A_nl
B_x_nl, B_y_nl, B_mag_nl = solver_nl_final.compute_flux_density()

# Create plots
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Magnetic potential
ax = fig.add_subplot(gs[0, :])
cs = ax.tricontourf(triang, A_nl, levels=30, cmap='coolwarm')
plt.colorbar(cs, ax=ax, label='A_z')
ax.set_aspect('equal')
ax.set_title('Magnetic Potential with Iron Ring', fontsize=14, weight='bold')

# 2. Flux density
ax = fig.add_subplot(gs[1, 0])
B_nodes_nl = np.zeros(mesh.n_nodes)
for e in range(mesh.n_elements):
    for node in mesh.elements[e]:
        B_nodes_nl[node] = max(B_nodes_nl[node], B_mag_nl[e])
cs = ax.tricontourf(triang, B_nodes_nl, levels=20, cmap='viridis')
plt.colorbar(cs, ax=ax, label='|B|')
ax.set_aspect('equal')
ax.set_title('Flux Density |B|', fontsize=12, weight='bold')

# 3. Flux lines
ax = fig.add_subplot(gs[1, 1])
interp_Bx_nl = LinearNDInterpolator(centers, B_x_nl)
interp_By_nl = LinearNDInterpolator(centers, B_y_nl)
Bx_grid_nl = np.nan_to_num(interp_Bx_nl(X, Y))
By_grid_nl = np.nan_to_num(interp_By_nl(X, Y))
ax.streamplot(X, Y, Bx_grid_nl, By_grid_nl, color='black', linewidth=1, density=1.5)
ax.set_aspect('equal')
ax.set_title('Flux Lines (note bending into iron)', fontsize=12, weight='bold')
ax.set_xlim(0, 2)
ax.set_ylim(0, 2)

# 4. μ distribution
ax = fig.add_subplot(gs[1, 2])
mu_nodes = np.zeros(mesh.n_nodes)
for e in range(mesh.n_elements):
    for node in mesh.elements[e]:
        mu_nodes[node] = max(mu_nodes[node], nl_solver.mu_current[e])
cs = ax.tricontourf(triang, mu_nodes, levels=20, cmap='RdYlGn')
plt.colorbar(cs, ax=ax, label='μ')
ax.set_aspect('equal')
ax.set_title('Permeability Distribution', fontsize=12, weight='bold')

# 5. Newton convergence
ax = fig.add_subplot(gs[2, 0])
ax.semilogy(residuals_nl, 'o-', linewidth=2, markersize=8, color='purple')
ax.axhline(1e-6, color='red', linestyle='--', label='Tolerance')
ax.set_xlabel('Newton Iteration', fontsize=12)
ax.set_ylabel('||ΔA|| (log scale)', fontsize=12)
ax.set_title('Newton-Raphson Convergence', fontsize=12, weight='bold')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# 6. μ evolution
ax = fig.add_subplot(gs[2, 1])
mu_history = [hist['mu'][is_iron].mean() for hist in nl_solver.iteration_history]
ax.plot(mu_history, 'o-', linewidth=2, markersize=8, color='green')
ax.set_xlabel('Newton Iteration', fontsize=12)
ax.set_ylabel('Average μ in Iron', fontsize=12)
ax.set_title('μ Evolution', fontsize=12, weight='bold')
ax.grid(True, alpha=0.3)

# 7. Operating point
ax = fig.add_subplot(gs[2, 2])
B_history = [hist['B_mag'][is_iron].mean() for hist in nl_solver.iteration_history]
ax.plot(H_range, B_range, linewidth=2, color='gray', label='Material curve')
H_traj = [B_history[i] / mu_history[i] for i in range(len(mu_history))]
ax.plot(H_traj, B_history, 'ro-', linewidth=2, markersize=6, label='Iteration path')
ax.plot(H_traj[-1], B_history[-1], 'g*', markersize=20, label='Final point')
ax.set_xlabel('H', fontsize=12)
ax.set_ylabel('B', fontsize=12)
ax.set_title('Operating Point (avg in iron)', fontsize=12, weight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('CASE 2: Coil with Ferromagnetic Ring (Nonlinear)', fontsize=16, weight='bold')
plt.show()

print("\n✅ Nonlinear solution visualized!")

---
<a id='section7'></a>
# 7. Linear vs Nonlinear Comparison

Compare CASE 1 (air) vs CASE 2 (iron ring)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Row 1: Linear (air)
ax = axes[0, 0]
cs = ax.tricontourf(triang, solver_direct.A, levels=20, cmap='coolwarm')
plt.colorbar(cs, ax=ax)
ax.set_aspect('equal')
ax.set_title('CASE 1: Potential (Air)', fontsize=12, weight='bold')

ax = axes[0, 1]
cs = ax.tricontourf(triang, B_nodes, levels=20, cmap='viridis')
plt.colorbar(cs, ax=ax)
ax.set_aspect('equal')
ax.set_title('CASE 1: |B| (Air)', fontsize=12, weight='bold')

ax = axes[0, 2]
ax.streamplot(X, Y, Bx_grid, By_grid, color='black', linewidth=1, density=1.5)
ax.set_aspect('equal')
ax.set_title('CASE 1: Flux Lines (Air)', fontsize=12, weight='bold')
ax.set_xlim(0, 2)
ax.set_ylim(0, 2)

# Row 2: Nonlinear (iron ring)
ax = axes[1, 0]
cs = ax.tricontourf(triang, A_nl, levels=20, cmap='coolwarm')
plt.colorbar(cs, ax=ax)
ax.set_aspect('equal')
ax.set_title('CASE 2: Potential (Iron Ring)', fontsize=12, weight='bold')

ax = axes[1, 1]
cs = ax.tricontourf(triang, B_nodes_nl, levels=20, cmap='viridis')
plt.colorbar(cs, ax=ax)
ax.set_aspect('equal')
ax.set_title('CASE 2: |B| (Iron Ring)', fontsize=12, weight='bold')

ax = axes[1, 2]
ax.streamplot(X, Y, Bx_grid_nl, By_grid_nl, color='black', linewidth=1, density=1.5)
ax.set_aspect('equal')
ax.set_title('CASE 2: Flux Lines (Iron Ring)', fontsize=12, weight='bold')
ax.set_xlim(0, 2)
ax.set_ylim(0, 2)

plt.suptitle('Comparison: Linear vs Nonlinear', fontsize=16, weight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Comparison complete!")
print(f"\nKey observations:")
print(f"  1. Max |B| in air: {B_mag.max():.4f}")
print(f"  2. Max |B| with iron: {B_mag_nl.max():.4f}")
print(f"  3. Iron concentrates flux (higher B in iron region)")
print(f"  4. Flux lines bend toward high-μ path (iron ring)")
print(f"  5. This is why motors/transformers use iron cores!")

---
<a id='section8'></a>
# 8. Summary & Extensions

## What We Learned

✅ **Theory:**
- Maxwell's equations → magnetostatic PDE
- Weak form and Galerkin method
- FEA discretization with linear triangles

✅ **Implementation:**
- Built FEA from scratch (assembly, BCs, solve)
- Direct solver (LU)
- Iterative solvers (Jacobi, GS, CG)
- Newton-Raphson for nonlinear problems

✅ **Physics:**
- **CASE 1:** Coil in air (linear, constant μ)
- **CASE 2:** Coil with iron ring (nonlinear, μ(B), saturation)
- Flux concentration in high-μ materials

## Connection to Commercial Tools

**FEMM, COMSOL, Ansys Maxwell** use the same principles:
- Same element matrices (K, F)
- Same Newton iteration for nonlinearity
- More advanced:
  - 3D (tetrahedral elements)
  - Higher-order elements (quadratic, cubic)
  - Adaptive meshing
  - Advanced solvers (multigrid, GPU)
  - Transient analysis (eddy currents)
  - Motion (motors)

## Possible Extensions

1. **3D:** Use tetrahedral elements
2. **Transient:** Add ∂A/∂t term for eddy currents
3. **Motion:** Rotating mesh for motors
4. **Circuit coupling:** Solve FEA + circuit simultaneously
5. **Optimization:** Topology optimization for core shape

---

## 🎉 Congratulations!

You now understand FEA from Maxwell's equations to working code!

**You can:**
- ✅ Implement FEA solvers from scratch
- ✅ Choose the right solver (direct vs iterative)
- ✅ Handle nonlinear materials
- ✅ Understand commercial FEA software
- ✅ Design electromagnetic devices

**Keep exploring!**